# Higher-Order Aberrated Gaussian Diagnostics

This notebook focuses on smooth aberrated lens phases. It measures how non-quadratic a strong Krivanek phase is over already-propagated Gaussian supports, then checks how a real-space Gaussian split reduces the local residual.


## Setup

This diagnostic also forces JAX onto CPU because the sampled residual fit uses many small least-squares solves. Restart the kernel before running if JAX has already been imported.


In [ ]:
import os
import sys

if "jax" in sys.modules:
    raise RuntimeError(
        "Restart the kernel and run this notebook from the top. "
        "This notebook sets JAX_PLATFORM_NAME=cpu before importing JAX."
    )

os.environ.setdefault("JAX_ENABLE_X64", "1")
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from temgym_core.aberrations import KrivanekCoeffs
from temgym_core.components import KrivanekLens
from temgym_core.gaussian import FreeSpacePropagator
from temgym_core.gaussian_corrections import (
    estimate_width_scale_for_residual,
    sampled_quadratic_residual_on_beam,
    split_gaussian_realspace,
)
from temgym_core.source import circular_input_wave

jax.config.update("jax_enable_x64", True)
print("JAX backend:", jax.devices()[0].platform)


## Aberrated Lens Residual And Splitting Check

This local check measures how non-quadratic a strong Krivanek phase is over the current Gaussian supports, then shows how a 5x5 split reduces the worst local residual. It is intentionally separate from the atom notebook so we can extend it with FFT probe comparisons next.


In [ ]:
voltage_lens = 300_000
aperture_radius = 0.15e-6
focal_length = 5e-3

coeffs = KrivanekCoeffs(
    C21=300.0,
    phi21=0.25 * jnp.pi,
    C23=120.0,
    phi23=-0.1 * jnp.pi,
    C32=150.0,
    phi32=0.4 * jnp.pi,
    C43=150.0,
    phi43=0.2 * jnp.pi,
)
lens = KrivanekLens(z=focal_length, focal_length=focal_length, coeffs=coeffs)
rays_in = circular_input_wave(
    aperture_radius=aperture_radius,
    waist=0.05e-6,
    voltage=voltage_lens,
    overlap_factor=2.0,
)
rays_at_lens = FreeSpacePropagator()(rays_in, focal_length)

residuals = []
for i in range(int(np.asarray(rays_at_lens.x).size)):
    ray_i = rays_at_lens[i]
    residual = sampled_quadratic_residual_on_beam(
        lambda xy: lens.phase_shift(xy),
        ray_i,
        scale=ray_i.k,
        support_radius=1.0,
        fit_samples_per_axis=5,
    )
    residuals.append(float(residual.max_abs))
residuals = np.asarray(residuals)

worst_idx = int(np.argmax(residuals))
worst_children = split_gaussian_realspace(
    rays_at_lens[worst_idx],
    grid_shape=(5, 5),
    support_radius=1.1,
    child_width_scale=0.6,
)
child_residuals = []
for i in range(int(np.asarray(worst_children.x).size)):
    child_i = worst_children[i]
    residual = sampled_quadratic_residual_on_beam(
        lambda xy: lens.phase_shift(xy),
        child_i,
        scale=child_i.k,
        support_radius=1.0,
        fit_samples_per_axis=5,
    )
    child_residuals.append(float(residual.max_abs))
child_residuals = np.asarray(child_residuals)

print(f"Input beams: {np.asarray(rays_in.x).size}")
print(
    "Parent residual rad median/p90/max:",
    np.median(residuals),
    np.quantile(residuals, 0.9),
    np.max(residuals),
)
print(
    "Worst split child residual rad median/p90/max:",
    np.median(child_residuals),
    np.quantile(child_residuals, 0.9),
    np.max(child_residuals),
)
print(
    "Estimated width scale for 0.25 rad target:",
    float(estimate_width_scale_for_residual(np.max(residuals), target=0.25)),
)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(residuals, bins=20, alpha=0.7, label="Parent beams")
ax.hist(child_residuals, bins=12, alpha=0.7, label="Worst beam children")
ax.axvline(0.25, color="k", linestyle="--", label="0.25 rad target")
ax.set_xlabel("local best-fit quadratic residual (rad)")
ax.set_ylabel("count")
ax.legend()
fig.tight_layout()
plt.show()